# How trufl ranks areas

> Why priority is a judgement, and what weights and gaps do to it

::: {.callout-note appearance="simple" icon=false}
**Explanation: understand why.** Read it away from the work. There is nothing to run. Want to act? Use a [how-to guide](../how-to/02_use_your_data.ipynb). [About the four kinds of page](../explanation/06_documentation.ipynb).
:::

There is no correct order in which to sample a contaminated landscape. An area can deserve a visit because it is probably bad, because nobody has been there or because many people live on it. These reasons conflict, and no measurement settles the conflict.

Multiple-criteria decision-making (MCDM) is a family of methods for such situations. It does not remove the judgement. It writes the judgement down as directions and weights, then applies it the same way to every area.

## A weight is an opinion with consequences

Consider four areas. Areas 0 and 1 have high values. Areas 1 and 3 have been sampled often.

In [ ]:
#| hide
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
from fastcore.test import test_eq

from trufl.utils import gridder
from trufl.callbacks import State, MaxCB, MinCB, StdCB, CountCB, PriorCB
from trufl.optimizer import Optimizer
from trufl.sampler import Sampler, rank_to_sample
from trufl.collector import DataCollector

fname_raster = '../files/ground-truth-01-4326-simulated.tif'
areas = gridder(fname_raster)
no_samples = gpd.GeoDataFrame(index=pd.Index([], name='loc_id'), geometry=None, data={'value': None})
prior_state = State(no_samples, areas, cbs=[PriorCB(fname_raster)])()
first_ranks = Optimizer(prior_state).get_rank(is_benefit_x=[True], w_vector=[1], s_method='CP')
first_n = rank_to_sample(first_ranks['rank'].reindex(areas.index).to_numpy(), budget=600, min=1, policy='quantiles')
measurements = DataCollector(fname_raster).collect(
    Sampler(areas).sample(first_n, method='uniform', rng=np.random.default_rng(0)))
state = State(measurements, areas, cbs=[MaxCB(), MinCB(), StdCB(), CountCB(), PriorCB(fname_raster)])()
ranks = Optimizer(state).get_rank(is_benefit_x=[True, True, True, False, True],
                                  w_vector=[0.25, 0.125, 0.125, 0.25, 0.25], n_method='LINEAR1', s_method='CP')

small = pd.DataFrame({'Max': [9.0, 9.5, 3.0, 2.5], 'Min': [2.0, 1.5, 0.5, 1.0],
                      'Standard Deviation': [2.5, 2.0, 0.8, 0.3], 'Count': [2, 12, 3, 10],
                      'Prior': [6.0, 7.0, 2.0, 1.5]}, index=pd.Index(range(4), name='area'))

def small_rank(df, weights, s_method='CP'):
    return Optimizer(df).get_rank(is_benefit_x=[True, True, True, False, True], w_vector=weights,
                                  n_method='LINEAR1', s_method=s_method)['rank'].sort_index()

balanced, count_heavy = [0.25, 0.125, 0.125, 0.25, 0.25], [0.1, 0.05, 0.05, 0.7, 0.1]
never = small.astype(float)
never.loc[3, ['Max', 'Min', 'Standard Deviation', 'Count']] = np.nan

In [ ]:
#| echo: false
small

,Max,Min,Standard Deviation,Count,Prior
area,,,,,
0,9.0,2.0,2.5,2,6.0
1,9.5,1.5,2.0,12,7.0
2,3.0,0.5,0.8,3,2.0
3,2.5,1.0,0.3,10,1.5


Two groups of people could look at this table and both be reasonable. One says that severity comes first. The other says that we should look where we know least, and puts most of the weight on the count.

In [ ]:
#| echo: false
views = pd.DataFrame({'Severity first': small_rank(small, balanced),
                      'Least known first': small_rank(small, count_heavy)})
test_eq(views.T.values.tolist(), [[1, 2, 3, 4], [1, 4, 2, 3]])
views

,Severity first,Least known first
area,,
0,1,1
1,2,4
2,3,2
3,4,3


Area 1 is second for the first group and last for the second. No data changed. This is why weights belong to the people who answer for the plan and not to whoever runs the code.

It also explains something that surprises people in a second round: the worst area often drops down the list. It has not become safer. It has become better known.

## Two ideas of "best"

Compromise programming asks how far an area is from being the most urgent on every criterion at once. TOPSIS also asks how far it is from being the least urgent. An area that is extreme on one criterion and ordinary on the others can fare differently under the two. With most weight on the count, they disagree about the last two places:

In [ ]:
#| echo: false
methods = pd.DataFrame({'Compromise programming': small_rank(small, count_heavy),
                        'TOPSIS': small_rank(small, count_heavy, 'TOPSIS')})
test_eq(methods.T.values.tolist(), [[1, 4, 2, 3], [1, 3, 2, 4]])
methods

,Compromise programming,TOPSIS
area,,
0,1,1
1,4,3
2,2,2
3,3,4


Neither is more correct. When a place in the list depends on the method, the evidence for that place is weak, and that is worth saying when the plan is presented.

## Why a gap counts as urgent

trufl treats a missing value as the most urgent one. The intention is cautious: not knowing is a reason to go and look. Here is area 3, the mildest of the four, if it had never been sampled:

In [ ]:
#| echo: false
gap = pd.DataFrame({'Sampled 10 times': small_rank(small, balanced), 'Never sampled': small_rank(never, balanced)})
test_eq(gap.T.values.tolist(), [[1, 2, 3, 4], [1, 3, 4, 2]])
gap

,Sampled 10 times,Never sampled
area,,
0,1,1
1,2,3
2,3,4
3,4,2


It rises from last to second on ignorance alone. For an unvisited area that may be what you want. But trufl cannot tell ignorance from accident. A hole in a survey or a blank typed into a spreadsheet receives the same promotion.

The opposite trap exists too. An area with a single sample has a standard deviation of zero, so it looks perfectly uniform when in truth almost nothing is known about its variation.

## A rank is not a risk

A rank compresses everything into an order. Ranks 1 and 2 may be nearly equal, or far apart, and equal areas still get different ranks. Above all, the order answers "where next?". Read as "where is it dangerous?", it is wrong, and the more successful the campaign, the more wrong it becomes, because well-measured hotspots sink down the list.